In [28]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [14]:
def llm(prompt):
    response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input=prompt
    )
    return response.output_text

In [15]:
llm("Hey, what's up?")

"Not much, just here to help with any questions or topics you'd like to discuss. How's your day going so far?"

In [16]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm not aware of a specific course you're referring to. Could you please provide more information about the course, such as its name or topic? Additionally, I can try to help you find out if it's still possible to join.


In [17]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [18]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [23]:
question = 'I just discovered the course. Can I join now?'
answer = llm(prompt)
print(answer)

You can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [1]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [2]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [3]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [5]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [6]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [7]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [11]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [12]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [13]:
search_results = search(question)

In [14]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [15]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [16]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [17]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [18]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [24]:
build_prompt(question, search_results)

'Question:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after sub

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [26]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [29]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=prompt
)

In [36]:
response.output[0]

ResponseReasoningItem(id='resp_01kv0vp5vtfjjrc07kd2z7zm2m', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed')

In [37]:
response.output[0].content[0].text

TypeError: 'NoneType' object is not subscriptable

In [41]:
print(response.output_text)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Additionally, you should be aware that to get a certificate, you need to finish the course with a "live" cohort and participate in peer-reviewing projects, which is only available during the course's runtime.


In [43]:
for item in response.output:
    print(item.type)

reasoning
message


In [44]:
for item in response.output:
    if item.type == "message":
        print(item.content[0].text)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Additionally, you should be aware that to get a certificate, you need to finish the course with a "live" cohort and participate in peer-reviewing projects, which is only available during the course's runtime.


In [42]:
response.usage

ResponseUsage(input_tokens=366, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=75, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=441)

In [46]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000612

In [48]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]

response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=message_history
)

In [50]:
print(response.output_text)

Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Keep in mind that you need to finish the course with a "live" cohort to be eligible for a certificate, as self-paced mode is not eligible for certification.


In [53]:
def llm(instructions, user_prompt, model='llama-3.3-70b-versatile'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
         ]
    
    response = openai_client.responses.create(
    model=model,
    input=message_history
    )

    return response.output_text

In [56]:
def rag(query, model='llama-3.3-70b-versatile'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer
    

In [57]:
answer = rag("I just discovered the course. Can i join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Note that you can start learning and submitting homework without formal registration, but to get a certificate, you must finish the course with a "live" cohort and participate in peer-reviewing projects.


In [59]:
print(rag("How do I get a certificate?"))

To get a certificate, you need to follow these steps:

1. **Finish the course with a "live" cohort**: You cannot get a certificate if you're taking the course in self-paced mode. You need to join a live cohort to be eligible for a certificate.
2. **Complete the Capstone project**: You must pass the Capstone project to get a certificate. Homework is not mandatory, but it's recommended to reinforce concepts.
3. **Peer-review 3 capstone projects**: After submitting your project, you need to peer-review 3 capstone projects from your cohort. This can only be done during the time the course is running.
4. **Ensure your official name is correct**: Make sure your official name is updated in your course profile, as this will appear on your certificate. You can do this by clicking on the "Edit Course Profile" button and filling in the second field with your official name.

By following these steps, you can earn a certificate upon completing the course with a live cohort.


In [60]:
print(rag("Which course is this?"))

This course appears to be "LLM-Zoomcamp", which is focused on Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG). The course has modules, homework, and a GitHub repository, and it offers a certificate upon completion with a "live" cohort.


In [61]:
print(rag("What's the requirement or prerequisites for this course?"))

The provided context doesn't explicitly state the requirements or prerequisites for the course. However, it does provide some general information about the course, such as:

* You don't need to pay for the Open AI API to complete the course homework.
* You can use free alternatives listed in the course GitHub.
* Registration is not necessary to start learning and submitting homework.
* You can only get a certificate if you finish the course with a "live" cohort.

To find the prerequisites or requirements for the course, you may want to check the course GitHub page or other resources provided by the course instructors.


In [62]:
print(rag("Which type of audience is targeted for this course?"))

The type of audience targeted for this course appears to be individuals with a technical background, likely in the fields of artificial intelligence, machine learning, or software development. This is inferred from the following:

1. The course topic: Introduction to LLMs (Large Language Models) and RAG ( Retrieval-Augmented Generation), which is a specialized topic in the field of AI.
2. The mention of OpenAI API, GitHub, and Slack, which suggests that the course is geared towards individuals who are familiar with these technologies.
3. The fact that the course has a peer-review component, which implies that the audience is expected to have a certain level of technical expertise to provide meaningful feedback.
4. The presence of a leaderboard, which suggests that the course is competitive and appealing to individuals who are motivated to learn and showcase their skills.

Overall, the target audience appears to be technical professionals, students, or enthusiasts who are interested in 

In [63]:
print(rag("What are the feature of Python programming language?"))

The provided context does not explicitly discuss the features of the Python programming language. However, based on general knowledge, the key features of Python include:

1. **Easy to Learn**: Python has a simple syntax and is relatively easy to learn, making it a great language for beginners.
2. **High-Level Language**: Python is a high-level language, meaning it abstracts away many low-level details, allowing developers to focus on the logic of their program without worrying about the underlying implementation.
3. **Object-Oriented**: Python supports object-oriented programming (OOP) concepts, such as classes, objects, inheritance, and polymorphism.
4. **Extensive Libraries**: Python has a vast collection of libraries and modules that make it suitable for a wide range of applications, including web development, data analysis, machine learning, and more.
5. **Cross-Platform**: Python can run on multiple operating systems, including Windows, macOS, and Linux.
6. **Large Community**: P